In [1]:
# load templates
from cobra import Metabolite, Reaction
from cobra.io import load_json_model, save_json_model
from modelseedpy.core.mstemplate import MSTemplateBuilder
from json import load
with open("../../../ModelSEEDTemplates/templates/v6.0/Core-V5.2.json") as fh:
    template_core = MSTemplateBuilder.from_dict(load(fh)).build()
with open("../../../ModelSEEDTemplates/templates/v6.0/GramNegModelTemplateV6.json") as fh:
    template_gramneg = MSTemplateBuilder.from_dict(load(fh)).build()
with open("../../../ModelSEEDTemplates/templates/v6.0/GramPosModelTemplateV6.json") as fh:
    template_grampos = MSTemplateBuilder.from_dict(load(fh)).build()

from modelseedpy import MSMedia
media = MSMedia.from_dict({'cpd00067': 1000.0,
 'cpd00058': 1000.0,
 'cpd00013': 1000.0,
 'cpd00244': 1000.0,
 'cpd00205': 1000.0,
 'cpd00034': 1000.0,
 'cpd11574': 1000.0,
 'cpd00971': 1000.0,
 'cpd00048': 1000.0,
 'cpd00030': 1000.0,
 'cpd00305': 100.0,
 'cpd00001': 1000.0,
 'cpd10516': 1000.0,
 'cpd00007': 1000.0,
 'cpd00159': 100.0,
 'cpd25960': 1000.0,
 'cpd00027': 10.0,
 "cpd00009": 100,
 'cpd00063': 1000.0,
 'cpd00149': 1000.0,
 'cpd00254': 1000.0,
 'cpd00099': 1000.0})

# load default medias
from modelseedpy.core.msatpcorrection import load_default_medias
default_medias = load_default_medias()
print(f'loaded {len(default_medias)} medias')

import math
def integrate_to_model_medium(mda, model, prefix='EX_'):
    medium = {}
    for cpd, (lb, ub) in mda.get_media_constraints().items():
        rxn_exchange = f'{prefix}{cpd}'
        if rxn_exchange in model.reactions:
            medium[rxn_exchange] = math.fabs(lb)
        else:
            print('not in model', cpd)
    return medium

# load models and correct ATP
from modelseedpy import MSATPCorrection
from modelseedpy import MSGapfill
betaine_models_paths = {                                                                                                                                          
      "models/Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.concoct_out.9.contigs__.RAST.json": "gram-pos",                                                         
      "models/Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.metabat.47.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.metabat.51.contigs__.RAST.json": "gram-pos",
      "models/Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.27.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.45.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.48.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.50.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_concoct_out.59.contigs__.RAST.json": "gram-pos",                                                        
      "models/Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.58.contigs__.RAST.json": "gram-neg",                                                            
      "models/Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_concoct_out.73.contigs__.RAST.json": "gram-pos",                                                        
      "models/Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_maxbin.047.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R1_C_D2_MG_DASTool_bins_concoct_out.85.contigs__.RAST.json": "gram-pos",                                                        
      "models/Salt_Pond_MetaG_R1_C_D2_MG_DASTool_bins_metabat.40.contigs__.RAST.json": "gram-pos",                                                            
      "models/Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.98.contigs__.RAST.json": "gram-neg",                                                        
      "models/Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.52.contigs__.RAST.json": "gram-pos",                                                            
  }

from os import path

def gapfill_model(model_path, template_label=None):
    if template_label is None:
        model_path, template_label = model_path
    template = template_gramneg if "neg" in template_label else template_grampos
    ## load model
    model = load_json_model(model_path)
    ## correct ATP
    atp_correction = MSATPCorrection(model, template_core, default_medias,
                                     compartment='c0', atp_hydrolysis_id='ATPM_c0', 
                                     load_default_medias=False)
    media_eval = atp_correction.evaluate_growth_media()
    atp_correction.determine_growth_media()
    atp_correction.apply_growth_media_gapfilling()
    atp_correction.expand_model_to_genome_scale()
    tests = atp_correction.build_tests()

    ## gapfill the model
    gapfill = MSGapfill(model, default_gapfill_templates=[template],
                        test_conditions=tests, default_target='bio1')
    gapfill_res = gapfill.run_gapfilling(media)
    gapfill.integrate_gapfill_solution(gapfill_res)
    print(f"Reactions in model after integration: {len(model.reactions)}")

    ## add missing exchange reactions for media compounds                                                                                                   
    for cpd, (lb, ub) in media.get_media_constraints().items():
        ex_id = f"EX_{cpd}"                                                                                                                                 
        if ex_id in model.reactions:  continue
        if cpd not in model.metabolites:                                                                                                                
            model.add_metabolites([Metabolite(cpd, compartment="e0")])                                                                                  
        ex_rxn = Reaction(ex_id)
        ex_rxn.lower_bound = -1000                                                                                                                      
        ex_rxn.upper_bound = 1000                     
        ex_rxn.add_metabolites({model.metabolites.get_by_id(cpd): -1})                                                                                  
        model.add_reactions([ex_rxn])
        print(f"  Added missing exchange: {ex_id}")    
   
    ## add gapfilling to the model's medium
    model.medium = integrate_to_model_medium(media, model)
    model.objective = 'bio1'
    display(model.summary())
    obj_val = model.slim_optimize()
    if obj_val > 0:
        print(f"Biomass flux: {obj_val}")
        save_json_model(model, model_path.replace('.json', '_gapfilled.json'))
    else:
        print(model_path, "failed to grow")

# for model_path, template_label in betaine_models_paths.items():
#     if path.exists(model_path.replace('.json', '_gapfilled.json')):
#         print(f"Model {model_path} already gapfilled")
#         continue
#     template = template_gramneg if "neg" in template_label else template_grampos
#     ## load model
#     model = load_json_model(model_path)
#     ## correct ATP
#     atp_correction = MSATPCorrection(model, template_core, default_medias,
#                                      compartment='c0', atp_hydrolysis_id='ATPM_c0', 
#                                      load_default_medias=False)
#     media_eval = atp_correction.evaluate_growth_media()
#     atp_correction.determine_growth_media()
#     atp_correction.apply_growth_media_gapfilling()
#     atp_correction.expand_model_to_genome_scale()
#     tests = atp_correction.build_tests()
#     new_tests = [t for t in tests if all(["." not in t['media'].id, "Glc" in t['media'].id])]
#     # for t in tests:
#     #     if "." not in t['media'].id:
#     #         new_tests.append(t)


#     ## gapfill the model
#     gapfill = MSGapfill(model, default_gapfill_templates=[template],
#                         test_conditions=new_tests, default_target='bio1')
#     gapfill_res = gapfill.run_gapfilling(media)
#     if gapfill_res is None:
#         print(f"Gapfilling failed for {model_path}")
#         gapfill = MSGapfill(model, default_gapfill_templates=[template],
#                         test_conditions=[], default_target='bio1')
#         gapfill_res = gapfill.run_gapfilling(media)
#     gapfill.integrate_gapfill_solution(gapfill_res)
#     print(f"Reactions in model after integration: {len(model.reactions)}")
   
#     ## add gapfilling to the model's medium
#     model.medium = integrate_to_model_medium(media, model)
#     model.objective = 'bio1'
#     display(model.summary())
#     obj_val = model.slim_optimize()
#     if obj_val > 0:
#         print(f"Biomass flux: {obj_val}")
#         save_json_model(model, model_path.replace('.json', '_gapfilled.json'))
#     else:
#         print(model_path, "failed to grow")

modelseedpy 0.4.3
loaded 54 medias


In [ ]:
args = [(model_path, template_label) for model_path, template_label in betaine_models_paths.items()
        if not path.exists(model_path.replace('.json', '_gapfilled.json'))]
print(len(args))
parallelize = False
if parallelize:
    from datetime import datetime
    from multiprocess import Pool
    from os import cpu_count

    pool_size = int(cpu_count()/2)
    print(f"Loading {pool_size} workers and computing the scores", datetime.now())
    pool = Pool(int(pool_size))  # .map(calculate_scores, [{k: v} for k,v in pairs.items()])
    output = pool.map(gapfill_model, args)
else:
    for model_path, template_label in args:
        gapfill_model(model_path, template_label)

7


No gapfilling solution found before filtering for Etho activating rxn00062_c0
No gapfilling solution found before filtering for mal-L activating rxn00062_c0
No gapfilling solution found before filtering for Pyr.SO4 activating rxn00062_c0
No gapfilling solution found before filtering for H2.SO4 activating rxn00062_c0
No gapfilling solution found before filtering for empty activating rxn00062_c0
No gapfilling solution found before filtering for Light activating rxn00062_c0
No gapfilling solution found before filtering for ANME activating rxn00062_c0
No gapfilling solution found before filtering for Methane activating rxn00062_c0
